## DFU Model Training - INAOE Dataset

This notebook implements model training using **preprocessed INAOE thermogram images**.

### Dataset
- **Source:** INAOE Plantar Thermogram Database (Preprocessed)
- **Total Images:** 334 (244 DM + 90 CT)
- **Status:** Ready to use (preprocessing already done)

### Model Architecture
- **EfficientNetB0**: Efficient mobile-friendly architecture
- **ResNet50**: Deep residual learning baseline
- **ConvNeXt-Tiny**: Modern convolutional neural network

### Training Strategy
- **Cross Validation:** 5-Fold stratified split
- **Two-Phase Training:** Frozen backbone → Fine-tuning
- **Hyperparameter Tuning:** Optuna (5 trials per model)
- **Loss Function:** Binary cross-entropy with class weighting
- **Threshold Optimization:** Sensitivity ≥ 85%

**Status:** Ready for training on INAOE preprocessed data

## 1. Setup & Imports

In [1]:
import os
import sys
import gc
import warnings
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import efficientnet, resnet50, convnext

import optuna
from optuna.trial import Trial
from optuna.samplers import TPESampler

# Suppress TensorFlow and absl warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF log levels
import logging
logging.getLogger('absl').setLevel(logging.ERROR)  # Suppress absl warnings

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Configure GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Configuration Parameters - USE INAOE PREPROCESSED DATASET
CONFIG = {
    'data_source': '/home/ntphoto/DFU/INAOE_Preprocessed',  # Use preprocessed dataset
    'checkpoint_dir': './model_checkpoints',
    'results_dir': './results',
    'img_size': (224, 224),
    'batch_size_default': 32,
    'max_epochs': 20,
    'phase1_patience': 5,
    'phase2_patience': 15,
    'n_folds': 5,
    'test_split': 0.2,
    'optuna_trials': 5,
    'phase2_lr_decay': 0.95,
    'phase2_decay_steps': 10000,
}

# Create output directories
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)

# Initialize logging
log_file = os.path.join(CONFIG['results_dir'], f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

def log_message(message: str, print_also: bool = True):
    """Log message to file and optionally to console"""
    with open(log_file, 'a') as f:
        f.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}\n")
    if print_also:
        print(message)

log_message("="*80)
log_message("DFU MODEL TRAINING - INAOE DATASET")
log_message(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_message(f"Data Source: {CONFIG['data_source']}")
log_message("="*80)

I0000 00:00:1776890343.516508   50902 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776890345.172559   50902 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776890346.542839   50902 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/ntphoto/miniconda3/envs/tf_gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jup

DFU MODEL TRAINING - INAOE DATASET
Timestamp: 2026-04-23 03:39:07
Data Source: /home/ntphoto/DFU/INAOE_Preprocessed


W0000 00:00:1776890347.296888   50902 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 2. Load Preprocessed INAOE Dataset

In [2]:
# ============================================================
# LOAD PREPROCESSED INAOE DATASET
# ============================================================

def load_preprocessed_inaoe(data_dir: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load preprocessed INAOE images from NPY files.
    
    Expected structure:
    - data_dir/
        - DM/       (label=1)
            - *.npy
        - CT/       (label=0)
            - *.npy
    
    Returns:
        images: (N, 224, 224, 3) normalized array [0, 1]
        labels: (N,) binary labels (0: CT, 1: DM)
    """
    images = []
    labels = []
    
    for group_idx, group in enumerate(['CT', 'DM']):
        group_dir = os.path.join(data_dir, group)
        
        if not os.path.exists(group_dir):
            raise FileNotFoundError(f"Group directory not found: {group_dir}")
        
        # Load all NPY files
        npy_files = sorted([f for f in os.listdir(group_dir) if f.endswith('.npy')])
        
        for npy_file in npy_files:
            npy_path = os.path.join(group_dir, npy_file)
            img_array = np.load(npy_path)
            
            images.append(img_array)
            labels.append(group_idx)  # 0: CT, 1: DM
    
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    
    log_message(f"Loaded preprocessed INAOE dataset:")
    log_message(f"  - Total images: {len(images)}")
    log_message(f"  - Shape: {images.shape}")
    log_message(f"  - CT (label=0): {np.sum(labels == 0)}")
    log_message(f"  - DM (label=1): {np.sum(labels == 1)}")
    log_message(f"  - Value range: [{images.min():.4f}, {images.max():.4f}]")
    
    return images, labels

# Load data
log_message(f"\nLoading preprocessed images from: {CONFIG['data_source']}")
try:
    images, labels = load_preprocessed_inaoe(CONFIG['data_source'])
    log_message(f"✓ Data loading successful: {images.shape[0]} samples")
except Exception as e:
    log_message(f"✗ Error loading data: {str(e)}")
    raise

# Verify data integrity
assert images.shape[1:] == (224, 224, 3), "Image size mismatch"
assert np.min(images) >= 0 and np.max(images) <= 1, "Image normalization issue"
assert len(np.unique(labels)) == 2, "Expected binary classification"

log_message(f"\nData verification:")
log_message(f"  - Min/Max pixel values: {images.min():.4f} / {images.max():.4f}")
log_message(f"  - DM (1): {np.sum(labels == 1)} | CT (0): {np.sum(labels == 0)}")
log_message(f"  - Class balance: {np.sum(labels == 1) / len(labels) * 100:.2f}% DM")


Loading preprocessed images from: /home/ntphoto/DFU/INAOE_Preprocessed
Loaded preprocessed INAOE dataset:
  - Total images: 334
  - Shape: (334, 224, 224, 3)
  - CT (label=0): 90
  - DM (label=1): 244
  - Value range: [0.0000, 1.0000]
✓ Data loading successful: 334 samples

Data verification:
  - Min/Max pixel values: 0.0000 / 1.0000
  - DM (1): 244 | CT (0): 90
  - Class balance: 73.05% DM


## 3. Train/Val/Test Split Logic

In [3]:
def create_fold_splits(images: np.ndarray, labels: np.ndarray, n_splits: int = 5, 
                       test_split: float = 0.2, random_state: int = SEED):
    """
    Create 5-fold cross-validation splits with stratification.
    Each fold: 80% train/val, 20% held-out test.
    """
    # First split: 80% for CV, 20% for final test set
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    
    fold_indices = []
    test_indices = None
    
    # Get first split as test set
    for fold_idx, (train_val_idx, test_idx) in enumerate(skf.split(images, labels)):
        if fold_idx == 0:
            test_indices = test_idx
            break
    
    # Create 5 folds from remaining data (excluding test set)
    train_val_images = images[train_val_idx]
    train_val_labels = labels[train_val_idx]
    train_val_original_idx = train_val_idx
    
    # Create 5 folds with stratification
    skf_folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf_folds.split(train_val_images, train_val_labels)):
        # Map back to original indices
        train_original = train_val_original_idx[train_idx]
        val_original = train_val_original_idx[val_idx]
        
        fold_indices.append({
            'fold': fold_idx,
            'train_idx': train_original,
            'val_idx': val_original
        })
    
    return fold_indices, test_indices

# Create folds
log_message("\nCreating 5-fold cross-validation splits...")
fold_indices, test_indices = create_fold_splits(
    images, labels,
    n_splits=CONFIG['n_folds'],
    test_split=CONFIG['test_split'],
    random_state=SEED
)

# Prepare test set
X_test = images[test_indices]
y_test = labels[test_indices]

log_message(f"✓ Created {len(fold_indices)} folds")
log_message(f"  - Test set size: {len(y_test)} (labels: {np.bincount(y_test)})")

for fold_info in fold_indices[:3]:  # Show first 3 folds
    fold_num = fold_info['fold']
    train_size = len(fold_info['train_idx'])
    val_size = len(fold_info['val_idx'])
    log_message(f"  - Fold {fold_num}: train={train_size}, val={val_size}")

# Store original images for use in training
X_full = images.copy()
y_full = labels.copy()

log_message("✓ Data split creation complete")


Creating 5-fold cross-validation splits...
✓ Created 5 folds
  - Test set size: 67 (labels: [18 49])
  - Fold 0: train=213, val=54
  - Fold 1: train=213, val=54
  - Fold 2: train=214, val=53
✓ Data split creation complete


## 4. Custom Model Class with Two-Phase Training

In [4]:
class ExponentialDecayScheduler(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Custom learning rate schedule for Phase 2: lr_t = lr_0 * 0.95^(step/10000)"""
    
    def __init__(self, initial_lr: float, decay_rate: float = 0.95, decay_steps: int = 10000):
        self.initial_lr = initial_lr
        self.decay_rate = decay_rate
        self.decay_steps = decay_steps
    
    def __call__(self, step):
        return self.initial_lr * tf.pow(self.decay_rate, tf.cast(step, tf.float32) / self.decay_steps)
    
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "decay_rate": self.decay_rate,
            "decay_steps": self.decay_steps,
        }


class DFUModelTrainer:
    """Two-phase training strategy for transfer learning models"""
    
    def __init__(self, model_name: str, base_model, dropout_rate: float = 0.5, 
                 l2_reg: float = 1e-5, dense_units: Tuple[int, int] = (256, 64)):
        """
        Initialize model with custom head.
        
        Args:
            model_name: Name of the model architecture
            base_model: Pre-trained base model (without top layers)
            dropout_rate: Dropout rate for regularization
            l2_reg: L2 regularization coefficient
            dense_units: Tuple of (dense_layer1_units, dense_layer2_units)
        """
        self.model_name = model_name
        self.base_model = base_model
        self.dropout_rate = dropout_rate
        self.l2_reg = l2_reg
        self.dense_units = dense_units
        self.model = None
        self.history = None
        self.phase1_history = None
        self.phase2_history = None
    
    def build_model(self):
        """Build model with custom head"""
        # Freeze base model
        self.base_model.trainable = False
        
        # Build custom head
        inputs = layers.Input(shape=(224, 224, 3))
        x = self.base_model(inputs, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(self.dense_units[0], activation='relu',
                        kernel_regularizer=keras.regularizers.l2(self.l2_reg))(x)
        x = layers.Dropout(self.dropout_rate)(x)
        x = layers.Dense(self.dense_units[1], activation='relu',
                        kernel_regularizer=keras.regularizers.l2(self.l2_reg))(x)
        x = layers.Dropout(self.dropout_rate)(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)
        
        self.model = models.Model(inputs=inputs, outputs=outputs)
        return self.model
    
    def train_phase1(self, X_train, y_train, X_val, y_val, batch_size: int = 32,
                     optimizer: str = 'adam', learning_rate: float = 1e-3,
                     max_epochs: int = 100, patience: int = 5, verbose: int = 1):
        """
        Phase 1: Train custom head with frozen backbone
        
        Early stopping with patience=5 epochs
        """
        log_message(f"\n{'='*80}")
        log_message(f"PHASE 1: Training Custom Head (Frozen Backbone) - {self.model_name}")
        log_message(f"{'='*80}")
        
        # Compile with frozen backbone
        if optimizer.lower() == 'adam':
            opt = Adam(learning_rate=learning_rate)
        elif optimizer.lower() == 'rmsprop':
            opt = RMSprop(learning_rate=learning_rate)
        else:
            opt = SGD(learning_rate=learning_rate, momentum=0.9)
        
        self.model.compile(
            optimizer=opt,
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        # Class weights for balanced training
        class_weights = {
            0: 1.0 / (np.sum(y_train == 0) / len(y_train)),
            1: 1.0 / (np.sum(y_train == 1) / len(y_train))
        }
        
        # Callbacks
        checkpoint_path = os.path.join(
            CONFIG['checkpoint_dir'],
            f"{self.model_name}_phase1_fold_best.h5"
        )
        
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                checkpoint_path,
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train Phase 1
        self.phase1_history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            batch_size=batch_size,
            epochs=max_epochs,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=verbose
        )
        
        log_message(f"Phase 1 training complete. Best val_loss: {min(self.phase1_history.history['val_loss']):.6f}")
    
    def train_phase2(self, X_train, y_train, X_val, y_val, batch_size: int = 32,
                     optimizer: str = 'adam', learning_rate: float = 1e-4,
                     max_epochs: int = 100, patience: int = 15, verbose: int = 1):
        """
        Phase 2: Fine-tune with unfrozen top layers using exponential decay
        
        Learning rate schedule: lr_t = lr_0 × 0.95^(t/10000)
        Early stopping with patience=15 epochs (continuing from Phase 1)
        """
        log_message(f"\n{'='*80}")
        log_message(f"PHASE 2: Fine-tuning with Top Layers Unfrozen - {self.model_name}")
        log_message(f"{'='*80}")
        
        # Unfreeze top layers of backbone
        num_layers = len(self.base_model.layers)
        unfreeze_from = int(num_layers * 0.7)  # Unfreeze top 30% of layers
        
        for layer in self.base_model.layers[unfreeze_from:]:
            layer.trainable = True
        
        log_message(f"Unfroze {num_layers - unfreeze_from} layers (from layer {unfreeze_from})")
        
        # Compile with exponential decay
        lr_schedule = ExponentialDecayScheduler(
            initial_lr=learning_rate,
            decay_rate=CONFIG['phase2_lr_decay'],
            decay_steps=CONFIG['phase2_decay_steps']
        )
        
        if optimizer.lower() == 'adam':
            opt = Adam(learning_rate=lr_schedule)
        elif optimizer.lower() == 'rmsprop':
            opt = RMSprop(learning_rate=lr_schedule)
        else:
            opt = SGD(learning_rate=lr_schedule, momentum=0.9)
        
        self.model.compile(
            optimizer=opt,
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        # Class weights
        class_weights = {
            0: 1.0 / (np.sum(y_train == 0) / len(y_train)),
            1: 1.0 / (np.sum(y_train == 1) / len(y_train))
        }
        
        # Callbacks
        checkpoint_path = os.path.join(
            CONFIG['checkpoint_dir'],
            f"{self.model_name}_phase2_fold_best.h5"
        )
        
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                checkpoint_path,
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train Phase 2
        self.phase2_history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            batch_size=batch_size,
            epochs=max_epochs,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=verbose
        )
        
        log_message(f"Phase 2 training complete. Best val_loss: {min(self.phase2_history.history['val_loss']):.6f}")
        log_message(f"{'='*80}\n")
    
    def get_predictions(self, X):
        """Get model predictions"""
        return self.model.predict(X, verbose=0).flatten()
    
    def save_model(self, path: str):
        """Save model to disk in Keras 3 format"""
        # Change .h5 to .keras for native Keras format
        if path.endswith('.h5'):
            path = path.replace('.h5', '.keras')
        self.model.save(path)
        log_message(f"Model saved to {path}")

## 5. Hyperparameter Tuning with Optuna

In [ ]:
class OptunaHyperparameterTuner:
    """Optuna-based hyperparameter optimization for DFU models"""
    
    def __init__(self, model_name: str, base_model_fn, n_trials: int = 50):
        """
        Initialize tuner.
        
        Args:
            model_name: Name of the model
            base_model_fn: Function to create base model
            n_trials: Number of trials to run
        """
        self.model_name = model_name
        self.base_model_fn = base_model_fn
        self.n_trials = n_trials
        self.best_params = None
        self.best_value = 0
        self.study = None
    
    def objective(self, trial: optuna.Trial, X_full: np.ndarray, y_full: np.ndarray, fold_indices: list) -> float:
        
        """Optuna objective function"""
        
        # Define search space
        dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.7)
        l2_reg = trial.suggest_categorical('l2_reg', [1e-6, 1e-5, 1e-4])
        dense_units_1 = trial.suggest_categorical('dense_units_1', [128, 256, 512])
        dense_units_2 = trial.suggest_categorical('dense_units_2', [32, 64, 128])
        batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
        optimizer = trial.suggest_categorical('optimizer', ['adam', 'rmsprop', 'sgd'])
        phase1_lr = trial.suggest_float('phase1_lr', 1e-4, 1e-2, log=True)
        phase2_lr = trial.suggest_float('phase2_lr', 1e-5, 1e-3, log=True)
        
        try:
            fold_aucs = []
            
            # 2. วนลูป 5 Folds ตรงนี้
            for fold_idx, fold_info in enumerate(fold_indices):
                X_train = X_full[fold_info['train_idx']]
                y_train = y_full[fold_info['train_idx']]
                X_val   = X_full[fold_info['val_idx']]
                y_val   = y_full[fold_info['val_idx']]
                
                # Build model
                base_model = self.base_model_fn()
                trainer = DFUModelTrainer(
                    model_name=f"{self.model_name}_cv_fold{fold_idx+1}",
                    base_model=base_model,
                     dropout_rate=dropout_rate,
                    l2_reg=l2_reg,
                    dense_units=(dense_units_1, dense_units_2)
                )
                trainer.build_model()
                
                # Phase 1 training
                trainer.train_phase1(
                X_train, y_train, X_val, y_val,
                batch_size=batch_size,
                optimizer=optimizer,
                learning_rate=phase1_lr,
                max_epochs=20,
                patience=5,
                verbose=0
                )
            
                # Phase 2 training
                trainer.train_phase2(
                X_train, y_train, X_val, y_val,
                batch_size=batch_size,
                optimizer=optimizer,
                learning_rate=phase2_lr,
                max_epochs=20,
                patience=15,
                verbose=0
                )
            
                # Evaluate ของ Fold นี้
                val_preds = trainer.get_predictions(X_val)
                val_auc = roc_auc_score(y_val, val_preds)
                fold_aucs.append(val_auc)
                
                del trainer, base_model
                tf.keras.backend.clear_session()
                gc.collect()
            # 3. คืนค่าเฉลี่ยของ 5 folds กลับไปให้ Optuna
            return float(np.mean(fold_aucs))

        except Exception as e:
            log_message(f"Trial failed with error: {str(e)}")
            return 0.0
    
    def optimize(self, X_full: np.ndarray, y_full: np.ndarray, fold_indices: list):
        """Run Optuna optimization using Cross-Validation"""
    
        log_message(f"\n{'='*80}")
        log_message(f"OPTUNA HYPERPARAMETER TUNING: {self.model_name}")
        log_message(f"Number of trials: {self.n_trials}")
        log_message(f"{'='*80}")
    
        sampler = optuna.samplers.TPESampler(seed=42) # หรือ SEED ที่คุณตั้งไว้
        self.study = optuna.create_study(
        direction='maximize',
        sampler=sampler
    )
    
        # แก้ไขตรง lambda ให้ส่งค่าที่รับมาเข้าไปใน objective
        self.study.optimize(
        lambda trial: self.objective(trial, X_full, y_full, fold_indices),
        n_trials=self.n_trials,
        show_progress_bar=True
    )
    
        self.best_params = self.study.best_params
        self.best_value = self.study.best_value
        
        log_message(f"\n✓ Optimization complete")
        log_message(f"Best validation AUC: {self.best_value:.6f}")
        log_message(f"Best hyperparameters:")
        for key, value in self.best_params.items():
            log_message(f"  - {key}: {value}")
        
        return self.best_params


def create_base_models():
    """Create base models with ImageNet weights"""
    def efficientnet_b0():
        return efficientnet.EfficientNetB0(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    def create_resnet50():
        return resnet50.ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    def convnext_tiny():
        return convnext.ConvNeXtTiny(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
    
    return {
        'EfficientNetB0': efficientnet_b0,
        'ResNet50': create_resnet50,
        'ConvNeXt-Tiny': convnext_tiny
    }

base_model_creators = create_base_models()

log_message("✓ Optuna tuner and base model creators initialized")

✓ Optuna tuner and base model creators initialized


: 

## 6. Train EfficientNetB0 Model

In [ ]:
# ============================================================
# Training EfficientNetB0
# Resume-aware: skips Optuna if best_params exists,
# skips individual folds if their checkpoint exists.
# ============================================================

model_name_efficientnet = 'EfficientNetB0'
efficientnet_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_efficientnet}")
log_message(f"{'#'*80}\n")

checkpoint_paths_eff = [
    os.path.join(CONFIG['checkpoint_dir'], f"{model_name_efficientnet}_Fold{i+1}_final.keras")
    for i in range(CONFIG['n_folds'])
]
best_params_path_eff = os.path.join(CONFIG['checkpoint_dir'], f"{model_name_efficientnet}_best_params.json")

# ── STEP 1: Hyperparameters (skip if already saved) ───────
if os.path.exists(best_params_path_eff):
    with open(best_params_path_eff, 'r') as _f:
        best_params = json.load(_f)
    log_message(f"✓ Loaded existing best_params from {best_params_path_eff} — skipping Optuna\n")
else:
    log_message(f"--- STEP 1: Hyperparameter Tuning (5-FOLD CV) ---")
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_efficientnet}_HyperparameterSearch",
        base_model_fn=base_model_creators['EfficientNetB0'],
        n_trials=CONFIG['optuna_trials']
    )
    best_params = tuner.optimize(X_full, y_full, fold_indices)

    with open(best_params_path_eff, 'w') as _f:
        json.dump(best_params, _f, indent=2)
    log_message(f"✓ best_params saved to {best_params_path_eff}\n")

# ── STEP 2: Per-fold training (resume from existing checkpoints) ───
log_message(f"--- STEP 2: Per-Fold Training (ALL {CONFIG['n_folds']} FOLDS) ---\n")

for fold_num, fold_info in enumerate(fold_indices):
    ckpt_path = checkpoint_paths_eff[fold_num]
    val_idx   = fold_info['val_idx']
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]

    if os.path.exists(ckpt_path):
        log_message(f"✓ Fold {fold_num+1}: checkpoint found — loading from {ckpt_path}")
        model = tf.keras.models.load_model(ckpt_path)
        val_predictions = model.predict(X_val_fold, verbose=0).flatten()
        efficientnet_results['fold_models'].append(model)
        efficientnet_results['fold_histories'].append({'phase1': {}, 'phase2': {}})
        efficientnet_results['fold_val_predictions'].append(val_predictions)
        continue

    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num+1}/{len(fold_indices)}: {model_name_efficientnet}  [TRAINING]")
    log_message(f"{'='*80}")

    train_idx    = fold_info['train_idx']
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]

    log_message(f"Train: {len(X_train_fold)} | Val: {len(X_val_fold)}")

    base_model = base_model_creators['EfficientNetB0']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_efficientnet}_Fold{fold_num+1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    trainer.build_model()

    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )

    val_predictions = trainer.get_predictions(X_val_fold)
    efficientnet_results['fold_models'].append(trainer.model)
    efficientnet_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    efficientnet_results['fold_val_predictions'].append(val_predictions)

    trainer.save_model(ckpt_path)
    log_message(f"✓ Fold {fold_num+1} training complete — saved to {ckpt_path}")

# ── Post-Training Metrics Summary ─────────────────────────
log_message(f"\n────────────────────────────────────────────────────────────")
log_message(f"  {model_name_efficientnet} — Validation Metrics Per Fold")
log_message(f"────────────────────────────────────────────────────────────")
_fold_aucs, _fold_sens, _fold_spec = [], [], []
for _fn, (_fi, _vp) in enumerate(zip(fold_indices, efficientnet_results['fold_val_predictions'])):
    _y_val = y_full[_fi['val_idx']]
    _auc  = roc_auc_score(_y_val, _vp)
    _yb   = (_vp >= 0.5).astype(int)
    _tp   = int(np.sum((_yb == 1) & (_y_val == 1)))
    _fp   = int(np.sum((_yb == 1) & (_y_val == 0)))
    _fn2  = int(np.sum((_yb == 0) & (_y_val == 1)))
    _tn   = int(np.sum((_yb == 0) & (_y_val == 0)))
    _sens = _tp / (_tp + _fn2) if (_tp + _fn2) > 0 else 0.0
    _spec = _tn / (_tn + _fp)  if (_tn + _fp)  > 0 else 0.0
    _fold_aucs.append(_auc); _fold_sens.append(_sens); _fold_spec.append(_spec)
    log_message(f"  Fold {_fn+1}:  AUC-ROC={_auc:.4f}  Sensitivity={_sens:.4f}  Specificity={_spec:.4f}")
log_message(f"────────────────────────────────────────────────────────────")
log_message(f"  Mean :  AUC-ROC={np.mean(_fold_aucs):.4f}  Sensitivity={np.mean(_fold_sens):.4f}  Specificity={np.mean(_fold_spec):.4f}")
log_message(f"  Std  :  AUC-ROC={np.std(_fold_aucs):.4f}   Sensitivity={np.std(_fold_sens):.4f}   Specificity={np.std(_fold_spec):.4f}")
log_message(f"────────────────────────────────────────────────────────────\n")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_efficientnet} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

[I 2026-04-23 03:39:07,845] A new study created in memory with name: no-name-f86f5f2b-f3b1-434b-a144-4eceaca2b730



################################################################################
# TRAINING EfficientNetB0
################################################################################

--- STEP 1: Hyperparameter Tuning (5-FOLD CV) ---

OPTUNA HYPERPARAMETER TUNING: EfficientNetB0_HyperparameterSearch
Number of trials: 5


  0%|          | 0/5 [00:00<?, ?it/s]W0000 00:00:1776890347.898464   50902 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1776890347.984191   50902 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13358 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a



PHASE 1: Training Custom Head (Frozen Backbone) - EfficientNetB0_HyperparameterSearch_cv_fold1


I0000 00:00:1776890353.112844   52265 service.cc:153] XLA service 0x7f771806b6f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776890353.112868   52265 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5060 Ti, Compute Capability 12.0a (Driver: 13.1.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.14.0)
I0000 00:00:1776890353.257716   52265 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776890354.279755   52265 cuda_dnn.cc:461] Loaded cuDNN version 91400
I0000 00:00:1776890354.486623   52265 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_16862__.259
E0000 00:00:1776890358.435774   52265 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1776890361.870724   52265 dot_search_space.cc:240] All configs were

...

Restoring model weights from the end of the best epoch: 20.
Phase 2 training complete. Best val_loss: 0.645545


PHASE 1: Training Custom Head (Frozen Backbone) - EfficientNetB0_HyperparameterSearch_cv_fold3


I0000 00:00:1776891069.766831   52267 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_505175__.259
I0000 00:00:1776891077.756673   52263 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_505175__.259
I0000 00:00:1776891078.111386   82970 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_8', 20 bytes spill stores, 20 bytes spill loads

E0000 00:00:1776891090.266811   52263 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1776891091.373015   52263 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1776891094.872861   83130 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm

Epoch 12: early stopping
Restoring model weights from the end of the best epoch: 7.
Phase 1 training complete. Best val_loss: 0.681867

PHASE 2: Fine-tuning with Top Layers Unfrozen - EfficientNetB0_HyperparameterSearch_cv_fold3
Unfroze 72 layers (from layer 166)


I0000 00:00:1776891112.508852   52266 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_533950__.319
I0000 00:00:1776891124.764725   52264 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_533950__.319
I0000 00:00:1776891125.244325   85146 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_22', 12 bytes spill stores, 12 bytes spill loads



Restoring model weights from the end of the best epoch: 20.
Phase 2 training complete. Best val_loss: 0.656816



## 7. Train ResNet50 Model

In [ ]:
# ============================================================
# Training ResNet50
# Resume-aware: skips Optuna if best_params exists,
# skips individual folds if their checkpoint exists.
# ============================================================

model_name_resnet = 'ResNet50'
resnet_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_resnet}")
log_message(f"{'#'*80}\n")

checkpoint_paths_res = [
    os.path.join(CONFIG['checkpoint_dir'], f"{model_name_resnet}_Fold{i+1}_final.keras")
    for i in range(CONFIG['n_folds'])
]
best_params_path_res = os.path.join(CONFIG['checkpoint_dir'], f"{model_name_resnet}_best_params.json")

# ── STEP 1: Hyperparameters (skip if already saved) ───────
if os.path.exists(best_params_path_res):
    with open(best_params_path_res, 'r') as _f:
        best_params = json.load(_f)
    log_message(f"✓ Loaded existing best_params from {best_params_path_res} — skipping Optuna\n")
else:
    log_message(f"--- STEP 1: Hyperparameter Tuning (5-FOLD CV) ---")
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_resnet}_HyperparameterSearch",
        base_model_fn=base_model_creators['ResNet50'],
        n_trials=CONFIG['optuna_trials']
    )
    best_params = tuner.optimize(X_full, y_full, fold_indices)

    with open(best_params_path_res, 'w') as _f:
        json.dump(best_params, _f, indent=2)
    log_message(f"✓ best_params saved to {best_params_path_res}\n")

# ── STEP 2: Per-fold training (resume from existing checkpoints) ───
log_message(f"--- STEP 2: Per-Fold Training (ALL {CONFIG['n_folds']} FOLDS) ---\n")

for fold_num, fold_info in enumerate(fold_indices):
    ckpt_path  = checkpoint_paths_res[fold_num]
    val_idx    = fold_info['val_idx']
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]

    if os.path.exists(ckpt_path):
        log_message(f"✓ Fold {fold_num+1}: checkpoint found — loading from {ckpt_path}")
        model = tf.keras.models.load_model(ckpt_path)
        val_predictions = model.predict(X_val_fold, verbose=0).flatten()
        resnet_results['fold_models'].append(model)
        resnet_results['fold_histories'].append({'phase1': {}, 'phase2': {}})
        resnet_results['fold_val_predictions'].append(val_predictions)
        continue

    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num+1}/{len(fold_indices)}: {model_name_resnet}  [TRAINING]")
    log_message(f"{'='*80}")

    train_idx    = fold_info['train_idx']
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]

    log_message(f"Train: {len(X_train_fold)} | Val: {len(X_val_fold)}")

    base_model = base_model_creators['ResNet50']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_resnet}_Fold{fold_num+1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    trainer.build_model()

    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )

    val_predictions = trainer.get_predictions(X_val_fold)
    resnet_results['fold_models'].append(trainer.model)
    resnet_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    resnet_results['fold_val_predictions'].append(val_predictions)

    trainer.save_model(ckpt_path)
    log_message(f"✓ Fold {fold_num+1} training complete — saved to {ckpt_path}")

# ── Post-Training Metrics Summary ─────────────────────────
log_message(f"\n────────────────────────────────────────────────────────────")
log_message(f"  {model_name_resnet} — Validation Metrics Per Fold")
log_message(f"────────────────────────────────────────────────────────────")
_fold_aucs, _fold_sens, _fold_spec = [], [], []
for _fn, (_fi, _vp) in enumerate(zip(fold_indices, resnet_results['fold_val_predictions'])):
    _y_val = y_full[_fi['val_idx']]
    _auc  = roc_auc_score(_y_val, _vp)
    _yb   = (_vp >= 0.5).astype(int)
    _tp   = int(np.sum((_yb == 1) & (_y_val == 1)))
    _fp   = int(np.sum((_yb == 1) & (_y_val == 0)))
    _fn2  = int(np.sum((_yb == 0) & (_y_val == 1)))
    _tn   = int(np.sum((_yb == 0) & (_y_val == 0)))
    _sens = _tp / (_tp + _fn2) if (_tp + _fn2) > 0 else 0.0
    _spec = _tn / (_tn + _fp)  if (_tn + _fp)  > 0 else 0.0
    _fold_aucs.append(_auc); _fold_sens.append(_sens); _fold_spec.append(_spec)
    log_message(f"  Fold {_fn+1}:  AUC-ROC={_auc:.4f}  Sensitivity={_sens:.4f}  Specificity={_spec:.4f}")
log_message(f"────────────────────────────────────────────────────────────")
log_message(f"  Mean :  AUC-ROC={np.mean(_fold_aucs):.4f}  Sensitivity={np.mean(_fold_sens):.4f}  Specificity={np.mean(_fold_spec):.4f}")
log_message(f"  Std  :  AUC-ROC={np.std(_fold_aucs):.4f}   Sensitivity={np.std(_fold_sens):.4f}   Specificity={np.std(_fold_spec):.4f}")
log_message(f"────────────────────────────────────────────────────────────\n")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_resnet} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

## 8. Train ConvNeXt-Tiny Model

In [ ]:
# ============================================================
# Training ConvNeXt-Tiny
# Resume-aware: skips Optuna if best_params exists,
# skips individual folds if their checkpoint exists.
# ============================================================

model_name_convnext = 'ConvNeXt-Tiny'
convnext_results = {
    'fold_models': [],
    'fold_histories': [],
    'fold_val_predictions': [],
    'fold_best_thresholds': [],
    'fold_metrics': []
}

log_message(f"\n{'#'*80}")
log_message(f"# TRAINING {model_name_convnext}")
log_message(f"{'#'*80}\n")

checkpoint_paths_cnx = [
    os.path.join(CONFIG['checkpoint_dir'], f"{model_name_convnext}_Fold{i+1}_final.keras")
    for i in range(CONFIG['n_folds'])
]
best_params_path_cnx = os.path.join(CONFIG['checkpoint_dir'], f"{model_name_convnext}_best_params.json")

# ── STEP 1: Hyperparameters (skip if already saved) ───────
if os.path.exists(best_params_path_cnx):
    with open(best_params_path_cnx, 'r') as _f:
        best_params = json.load(_f)
    log_message(f"✓ Loaded existing best_params from {best_params_path_cnx} — skipping Optuna\n")
else:
    log_message(f"--- STEP 1: Hyperparameter Tuning (5-FOLD CV) ---")
    tuner = OptunaHyperparameterTuner(
        model_name=f"{model_name_convnext}_HyperparameterSearch",
        base_model_fn=base_model_creators['ConvNeXt-Tiny'],
        n_trials=CONFIG['optuna_trials']
    )
    best_params = tuner.optimize(X_full, y_full, fold_indices)

    with open(best_params_path_cnx, 'w') as _f:
        json.dump(best_params, _f, indent=2)
    log_message(f"✓ best_params saved to {best_params_path_cnx}\n")

# ── STEP 2: Per-fold training (resume from existing checkpoints) ───
log_message(f"--- STEP 2: Per-Fold Training (ALL {CONFIG['n_folds']} FOLDS) ---\n")

for fold_num, fold_info in enumerate(fold_indices):
    ckpt_path  = checkpoint_paths_cnx[fold_num]
    val_idx    = fold_info['val_idx']
    X_val_fold = X_full[val_idx]
    y_val_fold = y_full[val_idx]

    if os.path.exists(ckpt_path):
        log_message(f"✓ Fold {fold_num+1}: checkpoint found — loading from {ckpt_path}")
        model = tf.keras.models.load_model(ckpt_path)
        val_predictions = model.predict(X_val_fold, verbose=0).flatten()
        convnext_results['fold_models'].append(model)
        convnext_results['fold_histories'].append({'phase1': {}, 'phase2': {}})
        convnext_results['fold_val_predictions'].append(val_predictions)
        continue

    log_message(f"\n{'='*80}")
    log_message(f"FOLD {fold_num+1}/{len(fold_indices)}: {model_name_convnext}  [TRAINING]")
    log_message(f"{'='*80}")

    train_idx    = fold_info['train_idx']
    X_train_fold = X_full[train_idx]
    y_train_fold = y_full[train_idx]

    log_message(f"Train: {len(X_train_fold)} | Val: {len(X_val_fold)}")

    base_model = base_model_creators['ConvNeXt-Tiny']()
    trainer = DFUModelTrainer(
        model_name=f"{model_name_convnext}_Fold{fold_num+1}",
        base_model=base_model,
        dropout_rate=best_params['dropout_rate'],
        l2_reg=best_params['l2_reg'],
        dense_units=(best_params['dense_units_1'], best_params['dense_units_2'])
    )
    trainer.build_model()

    trainer.train_phase1(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    trainer.train_phase2(
        X_train_fold, y_train_fold, X_val_fold, y_val_fold,
        batch_size=best_params['batch_size'],
        optimizer=best_params['optimizer'],
        learning_rate=best_params['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )

    val_predictions = trainer.get_predictions(X_val_fold)
    convnext_results['fold_models'].append(trainer.model)
    convnext_results['fold_histories'].append({
        'phase1': trainer.phase1_history.history,
        'phase2': trainer.phase2_history.history
    })
    convnext_results['fold_val_predictions'].append(val_predictions)

    trainer.save_model(ckpt_path)
    log_message(f"✓ Fold {fold_num+1} training complete — saved to {ckpt_path}")

# ── Post-Training Metrics Summary ─────────────────────────
log_message(f"\n────────────────────────────────────────────────────────────")
log_message(f"  {model_name_convnext} — Validation Metrics Per Fold")
log_message(f"────────────────────────────────────────────────────────────")
_fold_aucs, _fold_sens, _fold_spec = [], [], []
for _fn, (_fi, _vp) in enumerate(zip(fold_indices, convnext_results['fold_val_predictions'])):
    _y_val = y_full[_fi['val_idx']]
    _auc  = roc_auc_score(_y_val, _vp)
    _yb   = (_vp >= 0.5).astype(int)
    _tp   = int(np.sum((_yb == 1) & (_y_val == 1)))
    _fp   = int(np.sum((_yb == 1) & (_y_val == 0)))
    _fn2  = int(np.sum((_yb == 0) & (_y_val == 1)))
    _tn   = int(np.sum((_yb == 0) & (_y_val == 0)))
    _sens = _tp / (_tp + _fn2) if (_tp + _fn2) > 0 else 0.0
    _spec = _tn / (_tn + _fp)  if (_tn + _fp)  > 0 else 0.0
    _fold_aucs.append(_auc); _fold_sens.append(_sens); _fold_spec.append(_spec)
    log_message(f"  Fold {_fn+1}:  AUC-ROC={_auc:.4f}  Sensitivity={_sens:.4f}  Specificity={_spec:.4f}")
log_message(f"────────────────────────────────────────────────────────────")
log_message(f"  Mean :  AUC-ROC={np.mean(_fold_aucs):.4f}  Sensitivity={np.mean(_fold_sens):.4f}  Specificity={np.mean(_fold_spec):.4f}")
log_message(f"  Std  :  AUC-ROC={np.std(_fold_aucs):.4f}   Sensitivity={np.std(_fold_sens):.4f}   Specificity={np.std(_fold_spec):.4f}")
log_message(f"────────────────────────────────────────────────────────────\n")

log_message(f"\n{'#'*80}")
log_message(f"# {model_name_convnext} TRAINING COMPLETE")
log_message(f"{'#'*80}\n")

## 9. Comparison Result

In [ ]:
print("="*80)
print("5-FOLD CV MEAN RESULTS COMPARISON")
print("="*80 + "\n")

criteria = {
    'auc_threshold': 0.8,
    'sensitivity_threshold': 0.85,
    'specificity_threshold': 0.7
}

print("Criteria:")
print(f"  - Mean AUC-ROC > {criteria['auc_threshold']}")
print(f"  - Mean Sensitivity > {criteria['sensitivity_threshold']}")
print(f"  - Mean Specificity > {criteria['specificity_threshold']}\n")

models_to_check = [
    (model_name_efficientnet, efficientnet_results),
    (model_name_resnet, resnet_results),
    (model_name_convnext, convnext_results)
]

def _compute_fold_means(results, eval_threshold=0.5):
    """Compute per-fold AUC / Sensitivity / Specificity from stored val predictions."""
    aucs, sens, spec = [], [], []
    for fold_info, val_preds in zip(fold_indices, results['fold_val_predictions']):
        y_val = y_full[fold_info['val_idx']]
        aucs.append(roc_auc_score(y_val, val_preds))
        y_bin = (val_preds >= eval_threshold).astype(int)
        tp = int(np.sum((y_bin == 1) & (y_val == 1)))
        fp = int(np.sum((y_bin == 1) & (y_val == 0)))
        fn = int(np.sum((y_bin == 0) & (y_val == 1)))
        tn = int(np.sum((y_bin == 0) & (y_val == 0)))
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(aucs)), float(np.mean(sens)), float(np.mean(spec))

model_evaluations = []
for name, results in models_to_check:
    if len(results.get('fold_val_predictions', [])) == 0:
        print(f"{name}: No fold predictions available.\n")
        continue

    mean_auc, mean_sens, mean_spec = _compute_fold_means(results)

    pass_auc  = mean_auc  > criteria['auc_threshold']
    pass_sens = mean_sens > criteria['sensitivity_threshold']
    pass_spec = mean_spec > criteria['specificity_threshold']
    qualifies = pass_auc and pass_sens and pass_spec

    print(f"{name} (5-Fold Mean):")
    print(f"  • Mean AUC-ROC:     {mean_auc:.4f} {'✓ PASS' if pass_auc else '✗ FAIL'}")
    print(f"  • Mean Sensitivity: {mean_sens:.4f} {'✓ PASS' if pass_sens else '✗ FAIL'}")
    print(f"  • Mean Specificity: {mean_spec:.4f} {'✓ PASS' if pass_spec else '✗ FAIL'}")
    print(f"  -> MEETS ALL CRITERIA: {'YES ✓' if qualifies else 'NO ✗'}\n")

    model_evaluations.append({
        'name':      name,
        'results':   results,
        'mean_auc':  mean_auc,
        'mean_sens': mean_sens,
        'mean_spec': mean_spec,
        'qualifies': qualifies
    })

# ── Auto-select best model ────────────────────────────────────────
# Rule: among models passing all 3 criteria, pick highest Mean AUC-ROC.
# If none passes, fall back to highest Mean AUC-ROC overall.
qualifying = [m for m in model_evaluations if m['qualifies']]
if qualifying:
    best = max(qualifying, key=lambda m: m['mean_auc'])
    selection_reason = f"passes all 3 criteria with highest Mean AUC-ROC ({best['mean_auc']:.4f})"
else:
    best = max(model_evaluations, key=lambda m: m['mean_auc'])
    selection_reason = f"no model met all criteria — fallback to highest Mean AUC-ROC ({best['mean_auc']:.4f})"

best_model_name    = best['name']
best_model_results = best['results']

print("="*80)
print(f"BEST MODEL SELECTED: {best_model_name}")
print(f"  Reason: {selection_reason}")
print("="*80)

## 10. Threshold Optimization

In [ ]:
def optimize_threshold_per_fold(y_true, y_pred, target_sensitivity=0.85):
    """
    Pick threshold whose sensitivity >= target and specificity is highest.
    Fallback: threshold closest to target sensitivity on the ROC curve.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)

    valid = []
    for i, th in enumerate(thresholds):
        if tpr[i] >= target_sensitivity:
            valid.append({
                'threshold':   float(th),
                'sensitivity': float(tpr[i]),
                'specificity': float(1 - fpr[i]),
            })

    if valid:
        optimal_threshold = max(valid, key=lambda x: x['specificity'])['threshold']
    else:
        idx = int(np.argmin(np.abs(tpr - target_sensitivity)))
        optimal_threshold = float(thresholds[idx])

    return optimal_threshold, fpr, tpr, thresholds

# ====================================================================
# Uses best_model_name / best_model_results chosen in Section 9.
# ====================================================================
log_message(f"\n{'='*80}")
log_message(f"THRESHOLD OPTIMIZATION (BEST MODEL: {best_model_name})")
log_message(f"{'='*80}\n")

# Reset to avoid double-appending if re-run
best_model_results['fold_best_thresholds'] = []

for fold_num, (fold_info, val_preds) in enumerate(zip(fold_indices, best_model_results['fold_val_predictions'])):
    y_val = y_full[fold_info['val_idx']]

    threshold, fpr, tpr, thresholds = optimize_threshold_per_fold(y_val, val_preds)
    best_model_results['fold_best_thresholds'].append(threshold)

    y_pred_binary = (val_preds >= threshold).astype(int)
    tp = int(np.sum((y_pred_binary == 1) & (y_val == 1)))
    fp = int(np.sum((y_pred_binary == 1) & (y_val == 0)))
    fn = int(np.sum((y_pred_binary == 0) & (y_val == 1)))
    tn = int(np.sum((y_pred_binary == 0) & (y_val == 0)))
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    log_message(f"  Fold {fold_num + 1}: Threshold={threshold:.4f}  Sensitivity={sensitivity:.4f}  Specificity={specificity:.4f}")

# Mean threshold across folds — applied on the test set in Section 11
mean_threshold = float(np.mean(best_model_results['fold_best_thresholds']))
std_threshold  = float(np.std(best_model_results['fold_best_thresholds']))
best_model_results['mean_threshold'] = mean_threshold

log_message(f"\n  Mean threshold across folds: {mean_threshold:.4f}  (std={std_threshold:.4f})")
log_message(f"\n✓ Threshold optimization complete for {best_model_name}")

## 11. Evaluation on Test Set

In [ ]:
# ============================================================
# FINAL EVALUATION
# 1) Re-train the selected best model on the FULL train set (trainval)
# 2) Evaluate on the held-out test set using the mean threshold
#    computed in Section 10.
# ============================================================
log_message(f"\n{'='*80}")
log_message(f"RE-TRAIN BEST MODEL ON FULL TRAIN SET  →  TEST-SET EVALUATION")
log_message(f"  Best model : {best_model_name}")
log_message(f"{'='*80}\n")

# ── STEP 1: Load best_params for the selected best model ─────────
best_params_path_final = os.path.join(CONFIG['checkpoint_dir'], f"{best_model_name}_best_params.json")
with open(best_params_path_final, 'r') as _f:
    best_params_final = json.load(_f)
log_message(f"✓ Loaded best_params from {best_params_path_final}")

# ── STEP 2: Build full train set (everything except test set) ────
trainval_mask = np.ones(len(X_full), dtype=bool)
trainval_mask[test_indices] = False
X_trainval = X_full[trainval_mask]
y_trainval = y_full[trainval_mask]

log_message(f"Full train set : {len(y_trainval)} samples  (DM={int(np.sum(y_trainval==1))}, CT={int(np.sum(y_trainval==0))})")
log_message(f"Test set       : {len(y_test)} samples  (DM={int(np.sum(y_test==1))}, CT={int(np.sum(y_test==0))})")
log_message(f"Mean threshold : {best_model_results['mean_threshold']:.4f}  (from Section 10)")

# ── STEP 3: Re-train from scratch on full train set ──────────────
final_ckpt_path = os.path.join(CONFIG['checkpoint_dir'], f"{best_model_name}_FINAL.keras")

if os.path.exists(final_ckpt_path):
    log_message(f"✓ Final checkpoint exists — loading: {final_ckpt_path}")
    final_model = tf.keras.models.load_model(final_ckpt_path)
else:
    log_message("Training final model on full train set...")

    # Small internal split for early-stopping validation
    from sklearn.model_selection import train_test_split
    X_tr, X_vl, y_tr, y_vl = train_test_split(
        X_trainval, y_trainval, test_size=0.1,
        stratify=y_trainval, random_state=SEED
    )

    base_model = base_model_creators[best_model_name]()
    final_trainer = DFUModelTrainer(
        model_name=f"{best_model_name}_FINAL",
        base_model=base_model,
        dropout_rate=best_params_final['dropout_rate'],
        l2_reg=best_params_final['l2_reg'],
        dense_units=(best_params_final['dense_units_1'], best_params_final['dense_units_2'])
    )
    final_trainer.build_model()

    final_trainer.train_phase1(
        X_tr, y_tr, X_vl, y_vl,
        batch_size=best_params_final['batch_size'],
        optimizer=best_params_final['optimizer'],
        learning_rate=best_params_final['phase1_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase1_patience'],
        verbose=1
    )
    final_trainer.train_phase2(
        X_tr, y_tr, X_vl, y_vl,
        batch_size=best_params_final['batch_size'],
        optimizer=best_params_final['optimizer'],
        learning_rate=best_params_final['phase2_lr'],
        max_epochs=CONFIG['max_epochs'],
        patience=CONFIG['phase2_patience'],
        verbose=1
    )
    final_trainer.save_model(final_ckpt_path)
    final_model = final_trainer.model
    log_message(f"✓ Final model saved to {final_ckpt_path}")

# ── STEP 4: Predict on test set and apply the mean threshold ─────
test_probabilities = final_model.predict(X_test, verbose=0).flatten()
mean_threshold_final = best_model_results['mean_threshold']
test_predictions = (test_probabilities >= mean_threshold_final).astype(int)

# ── STEP 5: Compute final metrics ────────────────────────────────
tp = int(np.sum((test_predictions == 1) & (y_test == 1)))
fp = int(np.sum((test_predictions == 1) & (y_test == 0)))
fn = int(np.sum((test_predictions == 0) & (y_test == 1)))
tn = int(np.sum((test_predictions == 0) & (y_test == 0)))

final_metrics = {
    'model':       best_model_name,
    'threshold':   mean_threshold_final,
    'accuracy':    float(accuracy_score(y_test, test_predictions)),
    'precision':   float(precision_score(y_test, test_predictions, zero_division=0)),
    'recall':      float(recall_score(y_test, test_predictions, zero_division=0)),
    'f1':          float(f1_score(y_test, test_predictions, zero_division=0)),
    'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    'ppv':         tp / (tp + fp) if (tp + fp) > 0 else 0.0,
    'npv':         tn / (tn + fn) if (tn + fn) > 0 else 0.0,
    'auc_roc':     float(roc_auc_score(y_test, test_probabilities)),
    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
}

# Persist on best_model_results for downstream use
best_model_results['final_model']         = final_model
best_model_results['test_metrics']        = final_metrics
best_model_results['test_probabilities']  = test_probabilities
best_model_results['test_predictions']    = test_predictions

log_message(f"\n{'='*80}")
log_message(f"FINAL TEST-SET RESULTS — {best_model_name}")
log_message(f"{'='*80}")
log_message(f"  Threshold used : {final_metrics['threshold']:.4f}  (mean across CV folds)")
log_message(f"  Accuracy       : {final_metrics['accuracy']:.4f}")
log_message(f"  Precision      : {final_metrics['precision']:.4f}")
log_message(f"  Sensitivity    : {final_metrics['sensitivity']:.4f}")
log_message(f"  Specificity    : {final_metrics['specificity']:.4f}")
log_message(f"  PPV            : {final_metrics['ppv']:.4f}")
log_message(f"  NPV            : {final_metrics['npv']:.4f}")
log_message(f"  F1-Score       : {final_metrics['f1']:.4f}")
log_message(f"  AUC-ROC        : {final_metrics['auc_roc']:.4f}")
log_message(f"  Confusion      : TP={tp}  FP={fp}  FN={fn}  TN={tn}")

# Save results to CSV
final_df = pd.DataFrame([final_metrics])
final_csv_path = os.path.join(CONFIG['results_dir'], f'{best_model_name}_final_test_results.csv')
final_df.to_csv(final_csv_path, index=False)
log_message(f"\n✓ Final test results saved to {final_csv_path}")
log_message(f"\n✓ Evaluation complete")